[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/lens_camera/lens_camera.ipynb)

# A Zoom Camera with Depth of Field

This example photographs a grid of points at three depths through a two-lens zoom. A point at the focused depth images to a sharp dot; a point nearer or farther spreads its light over a blur disc, larger the further it is from focus and the wider the aperture. Moving the rear lens zooms, moving the sensor refocuses, and tilting the sensor brings a slanted plane into focus.

An ideal thin lens is a collineation of space, a map from points to points, `Point + origin * (home & Point) / focal`, and its action on lines is the join of the images of their points: a train of lenses composes either way. The rays a scene point sends through the aperture form a cone, a quadric, the aperture's ball pulled back through the projection from the point onto the aperture plane by its adjugate. The train carries that cone to the image cone, a pullback through the inverse of its point map, and the sensor cuts the image cone in the point's blur conic. Nothing asks where the point focuses: the cone's vertex is wherever the collineation puts it.

In the matrix notation of projective geometry the lens reads as a $4\times4$ homography $H$, the aperture cone as $Q' = P^{\mathsf T} Q P$ for the projection $P$, and the image cone as $H^{-\mathsf T} Q' H^{-1}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from numga.gatype.traits import Versor
from examples.animation import save_figure, show_animation
from examples.optics.lens_camera import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
SensorPoint = Point.derive.containing("x")
SensorPlane = Plane.derive.within("y z w")
# Lines are joins of points.
Line = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
# A collineation maps points to points.
PointMap = ga.gatype((Point, Point))          # Point <- Point
LineMap = ga.gatype((Line, Line))
# A primal quadric maps a point to its polar plane.
Quadric = ga.gatype((Plane, Point))           # Plane <- Point
# A camera maps a scene point and a pupil point to a sensor point.
Camera = ga.gatype((Point, Point, Point))     # Point <- (Point, Point)


def unit(points: Point) -> Point:
    """The representative of each point with positive unit weight; a unit point is a versor."""
    return (points / (mv.w & points)).normalized().with_traits(Versor)


# Every element is built in its home frame: centred on the origin, in the plane x == 0.
origin = unit(mv.w.dual())                                                      # Point
home = mv.x                                                                     # Plane

## 1. The thin lens as a collineation

At its home position the lens maps points by adding the origin weighted by their incidence with its plane over the focal length, and lines by subtracting the origin joined with their crossing of its plane. A motor places it. The train's map on lines is the join of the images of points under its map on points: the two descriptions agree.

In [ ]:
def thin_lens(focal: np.ndarray) -> tuple[PointMap, LineMap]:
    """Thin lenses of the given focal lengths in the home plane, as collineations of points and
    as the induced maps on lines."""
    points = Point + origin * (home & Point) / focal                            # PointMap
    lines = Line - (origin & (Line ^ home)) / focal                             # LineMap
    return points, lines


def ball(radius: float) -> Quadric:
    """A ball of the given radius about the origin; its section with the home plane is the aperture rim."""
    return mv.x * (mv.x & Point) + mv.y * (mv.y & Point) + mv.z * (mv.z & Point) - mv.w * (mv.w & Point) * radius**2


def expose(
    scene: Point, subject: Point, focal: np.ndarray, placements: Motor, focus: Point, tilt: Motor, radius: float, rim: Point,
):
    """Place the lenses, focus the sensor, and carry each scene point's aperture cone to the sensor side.

    Returns the lens train's collineation, the camera map, the sensor frame, the image cone of
    every scene point, the element planes [front, rear, sensor], and a fan of rays from the
    subject through the aperture rim, as its points on successive planes [plane + 1, ray].
    """
    # Place the lenses [front, rear], then compose their point maps and their line maps.
    lens_points, lens_lines = thin_lens(focal)
    points = placements >> lens_points(placements << Point)
    lines = placements >> lens_lines(placements << Line)
    front, rear = lines[0], lines[1]
    front_plane, rear_plane = placements[0] >> home, placements[1] >> home
    collineation = points[1](points[0])                                         # PointMap
    train = rear(front)
    pupil_ball = placements[0] >> ball(radius)(placements[0] << Point)          # Quadric

    # The focus point's image places the sensor; tilt turns it about that image.
    image = unit(collineation(focus))                                           # Point
    frame = (image / origin).square_root() * tilt                               # Motor
    sensor = frame >> home                                                      # Plane
    camera = train(Point & Point) ^ sensor                                      # Camera

    # Project through each subject onto the pupil, then pull back its quadric.
    # Pull back once more through the inverse lens train to get the image cones.
    project = (scene & Point) ^ front_plane
    cone = project.adjugate()(pupil_ball(project))                              # Quadric
    back = collineation.inverse()                                               # PointMap
    cones = back.adjugate()(cone(back))                                         # Quadric

    rays = subject & (placements[0] >> rim)                                     # Line
    legs = stack([subject.broadcast_to(rays.shape), rays ^ front_plane,         # Point
                         front(rays) ^ rear_plane, rear(front(rays)) ^ sensor])
    return collineation, camera, frame, cones, stack([front_plane, rear_plane, sensor]), legs


def section(cone: Quadric, start: Point, frame: Motor, samples: int) -> Point:
    """Boundary of the sensor's section of a cone, traced from a point inside it along the sensor plane."""
    theta = np.linspace(0.0, 2 * np.pi, samples)
    across = frame >> ((mv.yz * (-theta / 2)).exp() >> mv.y.dual())
    a, b, c = across & cone(across), across & cone(start), start & cone(start)
    # Rounding-level negative at the vertex:
    root = (b * b - a * c).clip(0.0, np.inf).square_root()
    return start + across * ((root - b) / a)

## 2. Wide, tele, and a tilted sensor

The scene is a grid of points at three depths, each depth drawn in its own colour on the sensor. Focused on the middle depth, wide and then tele, the middle layer is sharp dots and the near and far layers blur discs; tele magnifies and blurs more. With the sensor tilted, the plane of sharp focus tilts with it, and one side of each layer comes into focus.

In [ ]:
# The stills: the rear lens wide and tele, the focus distance, the aperture radius, and the sensor's tilt.
wide_at, tele_at = 1.4, 1.8
focus = 2.2
aperture = 0.45
tilt = np.radians(25)

In [ ]:
wide = scenarios.setting(wide_at, -focus, 0.0, aperture)
tele = scenarios.setting(tele_at, -focus, 0.0, aperture)
tilted = scenarios.setting(tele_at, -focus, tilt, aperture)
save_figure(render.draw_stills((wide, tele, tilted)), "lens_camera");

## 3. Each blur disc, two ways

The sensor's section of a point's image cone, traced around from where the chief ray through the aperture's centre meets the sensor, lies on the conic obtained the short way: the aperture disc, a flat dual quadric, pushed through the camera's map from the pupil to the sensor for that point. A point at the focused depth images to a point, its section collapsing onto the chief-ray hit.

In [ ]:
collineation, camera, frame, cones, planes, _ = tilted
place_front = (mv.xw * (scenarios.FRONT_AT / 2)).exp()                       # [] Motor
sensor, centre = frame >> home, place_front >> origin                          # [] Plane, [] Point
# The aperture disc as a flat dual quadric in the front lens's plane.
across_y, across_z = mv.y.dual(), mv.z.dual()
unit_disc = across_y * (across_y & Plane) + across_z * (across_z & Plane) - origin * (origin & Plane)   # [] Point <- Plane
pupil = place_front >> (unit_disc * aperture**2 - origin * (origin & Plane) * (1.0 - aperture**2))(place_front << Plane)
far_corner = scenarios.SCENE[2, 4, 3]                                          # [] Point
start = (collineation(far_corner) & collineation(centre)) ^ sensor             # [] Point
boundary = section(cones[2, 4, 3], start, frame, 48)                          # [samples] Point
to_sensor = camera(far_corner)                                                 # [] Point <- Point
pushed = to_sensor(pupil(to_sensor.adjugate()))                                # [] Point <- Plane
section_dual = (frame << pushed(frame >> SensorPlane)).cast(SensorPoint.output_subspace)
hits = (frame << boundary).cast(SensorPoint.output_subspace)

In [ ]:
print("traced section against the pushed disc:", np.abs((hits & section_dual.solve(hits)).kernel).max())

## 4. Zooming and refocusing

The rear lens swings to zoom while the focus moves through the layers and the aperture opens and closes; below the sensor image, the side view shows the lens planes and a fan of rays from one subject through the aperture's rim, leg by leg.

In [ ]:
n_frames, frame_duration = scenarios.N_FRAMES, scenarios.FRAME_DURATION
show_animation(render.animate_camera(scenarios.motion(n_frames), scenarios.SCENE), "lens_camera", frame_duration)

In [ ]:
# checks
# The image cone's vertex is the collineation's image of each scene point; the traced blur section lies
# on the pushed aperture disc; a point at the focus depth images to a point.
np.testing.assert_allclose((collineation(scenarios.SCENE) & cones(collineation(scenarios.SCENE))).kernel, 0.0, atol=1e-9)
np.testing.assert_allclose((hits & section_dual.solve(hits)).kernel, 0.0, atol=1e-10)
collineation, camera, frame, cones, planes, _ = wide
start = (collineation(scenarios.SCENE[1, 2, 1]) & collineation(centre)) ^ (frame >> home)
np.testing.assert_allclose(unit(section(cones[1, 2, 1], start, frame, 48)).kernel, unit(start).broadcast_to((48,)).kernel, atol=1e-6)